In [8]:
import mysql.connector
import sqlite3
from mysql.connector import Error

In [9]:
import sqlite3
import pandas as pd

def execute_sql_query(db_file, query):
    """
    Выполняет SQL-запрос к локальной базе данных и возвращает результат в виде DataFrame.

    :param db_file: путь к файлу локальной базы данных SQLite
    :param query: SQL-запрос для выполнения
    :return: результат запроса в виде pandas DataFrame
    """
    # Подключаемся к базе данных
    conn = sqlite3.connect(db_file)
    
    try:
        # Выполняем запрос и возвращаем результат в виде DataFrame
        result_df = pd.read_sql_query(query, conn)
        return result_df
    except Exception as e:
        print(f"Произошла ошибка при выполнении запроса: {e}")
        return None
    finally:
        # Закрываем соединение
        conn.close()

# Пример использования:
# db_file = 'local_credit_database.db'
# query = 'SELECT * FROM member LIMIT 10;'
# result = execute_sql_query(db_file, query)
# print(result)

In [3]:
import pymysql
import sqlite3
import os

# Параметры подключения к удаленной базе
remote_config = {
    'host': 'relational.fel.cvut.cz',
    'port': 3306,
    'user': 'guest',
    'password': 'ctu-relational',
    'database': 'Hockey',
    'charset': 'utf8mb4',
    'cursorclass': pymysql.cursors.DictCursor
}

# Шаг 1: Подключение к удаленной базе и получение данных
print("Подключение к удаленной базе...")
remote_conn = pymysql.connect(**remote_config)
remote_cursor = remote_conn.cursor()

# Шаг 2: Находим игроков, удовлетворяющих условиям запроса
print("Поиск подходящих игроков...")
remote_cursor.execute("""
    SELECT DISTINCT playerID 
    FROM Scoring 
    WHERE year BETWEEN 1970 AND 2010 
    LIMIT 100
""")
scoring_players = [row['playerID'] for row in remote_cursor.fetchall()]

# Находим игроков с наградами и тренерским опытом
remote_cursor.execute("""
    SELECT DISTINCT m.playerID
    FROM Master m
    JOIN AwardsPlayers ap ON m.playerID = ap.playerID
    JOIN Coaches c ON m.coachID = c.coachID
    WHERE m.playerID IS NOT NULL 
      AND m.coachID IS NOT NULL
      AND m.playerID IN (%s)
""" % ','.join(['%s'] * len(scoring_players)), scoring_players)

valid_players = [row['playerID'] for row in remote_cursor.fetchall()]
print(f"Найдено {len(valid_players)} подходящих игроков")

# Ограничиваем до 10 игроков для создания компактной локальной копии
selected_players = valid_players[:10]
print(f"Выбрано игроков для локальной копии: {selected_players}")

# Получаем coachID для выбранных игроков
remote_cursor.execute("""
    SELECT DISTINCT coachID 
    FROM Master 
    WHERE playerID IN (%s) AND coachID IS NOT NULL
""" % ','.join(['%s'] * len(selected_players)), selected_players)

coach_ids = [row['coachID'] for row in remote_cursor.fetchall()]
print(f"Найдено coachID: {coach_ids}")

# Собираем данные для всех необходимых таблиц
data = {}

# Данные из Master
remote_cursor.execute("""
    SELECT * FROM Master 
    WHERE playerID IN (%s) 
    LIMIT 40
""" % ','.join(['%s'] * len(selected_players)), selected_players)
data['Master'] = remote_cursor.fetchall()

# Данные из AwardsPlayers
remote_cursor.execute("""
    SELECT * FROM AwardsPlayers 
    WHERE playerID IN (%s) 
    LIMIT 40
""" % ','.join(['%s'] * len(selected_players)), selected_players)
data['AwardsPlayers'] = remote_cursor.fetchall()

# Данные из Coaches
if coach_ids:
    remote_cursor.execute("""
        SELECT * FROM Coaches 
        WHERE coachID IN (%s) 
        LIMIT 40
    """ % ','.join(['%s'] * len(coach_ids)), coach_ids)
    data['Coaches'] = remote_cursor.fetchall()
else:
    data['Coaches'] = []

# Данные из Scoring
remote_cursor.execute("""
    SELECT * FROM Scoring 
    WHERE playerID IN (%s) AND year BETWEEN 1970 AND 2010 
    LIMIT 40
""" % ','.join(['%s'] * len(selected_players)), selected_players)
data['Scoring'] = remote_cursor.fetchall()

# Закрываем соединение с удаленной базой
remote_conn.close()

# Шаг 3: Создание локальной SQLite базы
print("\nСоздание локальной базы данных...")
if os.path.exists('hockey_local.db'):
    os.remove('hockey_local.db')

local_conn = sqlite3.connect('hockey_local.db')
local_cursor = local_conn.cursor()

# Создаем таблицы с такой же структурой
local_cursor.execute('''
CREATE TABLE Master (
    playerID TEXT,
    coachID TEXT,
    hofID TEXT,
    firstName TEXT,
    lastName TEXT NOT NULL,
    nameNote TEXT,
    nameGiven TEXT,
    nameNick TEXT,
    height TEXT,
    weight TEXT,
    shootCatch TEXT,
    legendsID TEXT,
    ihdbID TEXT,
    hrefID TEXT,
    firstNHL TEXT,
    lastNHL TEXT,
    firstWHA TEXT,
    lastWHA TEXT,
    pos TEXT,
    birthYear TEXT,
    birthMon TEXT,
    birthDay TEXT,
    birthCountry TEXT,
    birthState TEXT,
    birthCity TEXT,
    deathYear TEXT,
    deathMon TEXT,
    deathDay TEXT,
    deathCountry TEXT,
    deathState TEXT,
    deathCity TEXT
)
''')

local_cursor.execute('''
CREATE TABLE AwardsPlayers (
    playerID TEXT NOT NULL,
    award TEXT NOT NULL,
    year INTEGER NOT NULL,
    lgID TEXT,
    note TEXT,
    pos TEXT,
    PRIMARY KEY (playerID, award, year)
)
''')

local_cursor.execute('''
CREATE TABLE Coaches (
    coachID TEXT NOT NULL,
    year INTEGER NOT NULL,
    tmID TEXT NOT NULL,
    lgID TEXT,
    stint INTEGER NOT NULL,
    notes TEXT,
    g INTEGER,
    w INTEGER,
    l INTEGER,
    t INTEGER,
    postg TEXT,
    postw TEXT,
    postl TEXT,
    postt TEXT,
    PRIMARY KEY (coachID, year, tmID, stint)
)
''')

local_cursor.execute('''
CREATE TABLE Scoring (
    playerID TEXT,
    year INTEGER,
    stint INTEGER,
    tmID TEXT,
    lgID TEXT,
    pos TEXT,
    GP INTEGER,
    G INTEGER,
    A INTEGER,
    Pts INTEGER,
    PIM INTEGER,
    "+/-" TEXT,
    PPG TEXT,
    PPA TEXT,
    SHG TEXT,
    SHA TEXT,
    GWG TEXT,
    GTG TEXT,
    SOG TEXT,
    PostGP TEXT,
    PostG TEXT,
    PostA TEXT,
    PostPts TEXT,
    PostPIM TEXT,
    "Post+/-" TEXT,
    PostPPG TEXT,
    PostPPA TEXT,
    PostSHG TEXT,
    PostSHA TEXT,
    PostGWG TEXT,
    PostSOG TEXT
)
''')

local_conn.commit()

# Шаг 4: Вставка данных в локальную базу
def insert_data(table_name, rows):
    """Вставка данных в SQLite таблицу"""
    if not rows:
        return
    
    columns = list(rows[0].keys())
    columns_str = ', '.join([f'"{col}"' if '+' in col or '-' in col else col for col in columns])
    placeholders = ', '.join(['?' for _ in columns])
    sql = f'INSERT INTO {table_name} ({columns_str}) VALUES ({placeholders})'
    
    data_to_insert = []
    for row in rows:
        row_values = []
        for col in columns:
            value = row[col]
            if value is None:
                row_values.append(None)
            elif isinstance(value, (int, float)):
                row_values.append(value)
            else:
                row_values.append(str(value) if value is not None else None)
        data_to_insert.append(tuple(row_values))
    
    local_cursor.executemany(sql, data_to_insert)
    local_conn.commit()
    print(f"Вставлено {len(data_to_insert)} строк в таблицу {table_name}")

print("\nВставка данных в локальную базу:")
for table_name, rows in data.items():
    insert_data(table_name, rows)

# Шаг 5: Проверка работоспособности запроса
print("\nПроверка исходного запроса на локальной базе...")
query = """
SELECT 
    COUNT(DISTINCT m.playerID) AS total_hof_players_with_awards_and_coaching
FROM 
    Master m 
    JOIN AwardsPlayers ap ON m.playerID = ap.playerID 
    JOIN Coaches c ON m.coachID = c.coachID
WHERE 
    m.playerID IN (
        SELECT playerID 
        FROM Scoring 
        WHERE year BETWEEN 1970 AND 2010
    )
"""

local_cursor.execute(query)
result = local_cursor.fetchone()
print(f"Результат запроса: {result}")

if result and result[0] > 0:
    print(f"✅ Успешно! Запрос вернул ненулевой результат: {result[0]}")
    print(f"✅ Локальная база создана успешно в файле 'hockey_local.db'")
else:
    print("❌ Ошибка: запрос вернул нулевой результат")

# Закрываем соединение с локальной базой
local_conn.close()

print("\nСтатистика по таблицам в локальной базе:")
for table_name in ['Master', 'AwardsPlayers', 'Coaches', 'Scoring']:
    conn = sqlite3.connect('hockey_local.db')
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    print(f"{table_name}: {count} строк (макс. 40)")
    conn.close()

Подключение к удаленной базе...
Поиск подходящих игроков...
Найдено 12 подходящих игроков
Выбрано игроков для локальной копии: ['cashmwa01', 'cheevge01', 'croziro01', 'esposph01', 'goyetph01', 'greente01', 'howelha01', 'hullbo01', 'mckenjo01', 'staplpa01']
Найдено coachID: ['cashmwa01c', 'cheevge01c', 'croziro01c', 'esposph01c', 'goyetph01c', 'greente01c', 'howelha01c', 'hullbo01c', 'mckenjo01c', 'staplpa01c']

Создание локальной базы данных...

Вставка данных в локальную базу:
Вставлено 10 строк в таблицу Master
Вставлено 40 строк в таблицу AwardsPlayers
Вставлено 24 строк в таблицу Coaches
Вставлено 40 строк в таблицу Scoring

Проверка исходного запроса на локальной базе...
Результат запроса: (4,)
✅ Успешно! Запрос вернул ненулевой результат: 4
✅ Локальная база создана успешно в файле 'hockey_local.db'

Статистика по таблицам в локальной базе:
Master: 10 строк (макс. 40)
AwardsPlayers: 40 строк (макс. 40)
Coaches: 24 строк (макс. 40)
Scoring: 40 строк (макс. 40)


In [4]:
import sqlite3
import pandas as pd
from io import StringIO
import csv

def get_database_info_and_csv():
    # Подключаемся к локальной базе данных
    conn = sqlite3.connect('local_hockey_database.db')
    cursor = conn.cursor()

    # Получаем список таблиц
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    output = StringIO()
    writer = csv.writer(output)

    for table_name in tables:
        table_name = table_name[0]
        print(f"Обработка таблицы: {table_name}")

        # Получаем структуру таблицы (CREATE TABLE)
        cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
        create_table_sql = cursor.fetchone()[0]
        output.write(f"\n--- Структура таблицы {table_name} ---\n")
        output.write(f"{create_table_sql}\n")

        # Получаем и выводим все данные из таблицы
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", conn)
        output.write(f"\n--- Данные таблицы {table_name} ---\n")
        
        # Записываем заголовки
        writer.writerow(df.columns.tolist())
        # Записываем данные
        for row in df.itertuples(index=False, name=None):
            writer.writerow(row)
        output.write("\n")  # Добавляем пустую строку между таблицами

    conn.close()
    
    return output.getvalue()

# Вызов функции и вывод результата
result = get_database_info_and_csv()
print(result)

Обработка таблицы: AwardsCoaches
Обработка таблицы: AwardsMisc
Обработка таблицы: AwardsPlayers
Обработка таблицы: Coaches
Обработка таблицы: CombinedShutouts
Обработка таблицы: Goalies
Обработка таблицы: GoaliesSC
Обработка таблицы: GoaliesShootout
Обработка таблицы: HOF
Обработка таблицы: Master
Обработка таблицы: Scoring
Обработка таблицы: ScoringSC
Обработка таблицы: ScoringShootout
Обработка таблицы: ScoringSup
Обработка таблицы: SeriesPost
Обработка таблицы: TeamSplits
Обработка таблицы: TeamVsTeam
Обработка таблицы: Teams
Обработка таблицы: TeamsHalf
Обработка таблицы: TeamsPost
Обработка таблицы: TeamsSC
Обработка таблицы: abbrev

--- Структура таблицы AwardsCoaches ---
CREATE TABLE AwardsCoaches (
                coachID TEXT,
                award TEXT,
                year INTEGER,
                lgID TEXT,
                note TEXT,
                FOREIGN KEY (coachID) REFERENCES Coaches (coachID) ON DELETE CASCADE ON UPDATE CASCADE
            )

--- Данные таблицы Award

In [6]:
query = """
SELECT 
    COUNT(DISTINCT m.playerID) AS total_hof_players_with_awards_and_coaching
FROM 
    Master m 
    JOIN AwardsPlayers ap ON m.playerID = ap.playerID 
    JOIN Coaches c ON m.coachID = c.coachID
WHERE 
    m.playerID IN (
        SELECT playerID 
        FROM Scoring 
        WHERE year BETWEEN 1970 AND 2010
    )
"""

In [10]:
db_file = 'hockey_local.db'
result = execute_sql_query(db_file, query)
print(result)

   total_hof_players_with_awards_and_coaching
0                                           4
